# What does the noise actually change? Task space at low vs high training noise — v8

Notebook 14 showed that the angle between the *where* and *whether* axes **rises with training noise**, and
that the spread across seeds **shrinks** as noise increases. This notebook shows what that summary line looks
like in the state space: the five-seed task-space figure at the **lowest** and the **highest** training noise,
side by side.

**The design point, and the reason this is a separate notebook.** Every network here is **trained at its own
noise level but tested on identical clean trials**. If a network trained without noise were tested clean and
one trained with noise tested with noise, two things would differ at once and any difference in the
trajectories could not be attributed to the training. Holding the test set fixed and clean means the only
thing that varies is what the network learned.

Only two noise levels are used, not all seven, because between training noise 1.0 and 1.5 the summary
measures barely move, so the extremes carry the whole story without crowding the figure.

**Seed 4 gets its own figure.** At training noise 0 its two axes were only about 7 degrees apart, the most
collinear of the five seeds, so if noise pushes networks towards orthogonal codes then seed 4 is where the
change should be largest. Seed 0 started near 82 degrees and can only gain a few.

Requires `scikit-learn`. Trials are generated in-notebook so the training noise can be set freely.

## 1. Setup

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from pathlib import Path

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cpu")
OUT_DIR = Path("./noise_geometry_v8"); OUT_DIR.mkdir(exist_ok=True)
T = 50; T_ON, T_OFF = 10, 20

## 2. Trial generator

The v8 specification with the baseline noise as a knob. Used to build a **training** set at each noise
level, and one **clean** test set that every network is evaluated and projected on.

In [ ]:
SUBTASKS = ["det_absent","det_auditory_only","det_visual_only",
            "loc_auditory_only_L","loc_auditory_only_R","loc_visual_only_L","loc_visual_only_R",
            "loc_multisensory_same_L","loc_multisensory_same_R","det_multisensory",
            "loc_conflict_audL_visR","loc_conflict_audR_visL"]
CONFLICT = ["det_multisensory","loc_conflict_audL_visR","loc_conflict_audR_visL"]
NONCONF  = [s for s in SUBTASKS if s not in CONFLICT]
LOC_FIT  = ["loc_auditory_only_L","loc_auditory_only_R","loc_visual_only_L","loc_visual_only_R",
            "loc_multisensory_same_L","loc_multisensory_same_R"]
DET_FIT  = ["det_absent","det_auditory_only","det_visual_only"]
TRAIN_COUNTS = {"det_absent":600,"det_visual_only":600,"det_auditory_only":1200,
                "loc_auditory_only_L":300,"loc_auditory_only_R":300,"loc_visual_only_L":300,
                "loc_visual_only_R":300,"loc_multisensory_same_L":300,"loc_multisensory_same_R":300,
                "loc_conflict_audL_visR":300,"loc_conflict_audR_visL":300}   # det_multisensory test-only

def make_trial(sub, noise_std, rng, split):
    X = rng.normal(0.0, noise_std, (4, T)).astype(np.float32) if noise_std > 0 else np.zeros((4, T), np.float32)
    def add(ch, i): X[ch, T_ON:T_OFF] += i
    ai = vi = np.nan
    if   sub == "det_absent": lab = 0
    elif sub == "det_auditory_only": i = rng.uniform(1,3); add(0,i); add(1,i); lab = 1
    elif sub == "det_visual_only":   i = rng.uniform(1,3); add(2,i); add(3,i); lab = 0
    elif sub == "loc_auditory_only_L": add(0, rng.uniform(1,3)); lab = 3
    elif sub == "loc_auditory_only_R": add(1, rng.uniform(1,3)); lab = 2
    elif sub == "loc_visual_only_L":   add(2, rng.uniform(1,3)); lab = 3
    elif sub == "loc_visual_only_R":   add(3, rng.uniform(1,3)); lab = 2
    elif sub == "loc_multisensory_same_L": i = rng.uniform(1,3); add(0,i); add(2,i); lab = 3
    elif sub == "loc_multisensory_same_R": i = rng.uniform(1,3); add(1,i); add(3,i); lab = 2
    elif sub == "det_multisensory": i = rng.uniform(1,3); [add(c,i) for c in range(4)]; lab = 1
    elif sub == "loc_conflict_audL_visR":
        ai = rng.uniform(1,3); vi = rng.uniform(1,3); add(0, ai); add(3, vi)
        lab = (3 if ai > vi else 2) if split == "test" else int(rng.integers(2,4))
    elif sub == "loc_conflict_audR_visL":
        ai = rng.uniform(1,3); vi = rng.uniform(1,3); add(1, ai); add(2, vi)
        lab = (2 if ai > vi else 3) if split == "test" else int(rng.integers(2,4))
    return X, int(lab), np.float32(ai), np.float32(vi)

def make_split(split, noise_std, seed=0):
    rng = np.random.default_rng(1000 + seed + (0 if split == "train" else 7))
    Xs, ys, ts, A, V = [], [], [], [], []
    counts = TRAIN_COUNTS if split == "train" else {s: 100 for s in SUBTASKS}
    for sub, n in counts.items():
        for _ in range(n):
            X, lab, ai, vi = make_trial(sub, noise_std, rng, split)
            Xs.append(X); ys.append(lab); ts.append(sub); A.append(ai); V.append(vi)
    return {"X": np.stack(Xs), "y": np.array(ys, np.int64), "types": np.array(ts),
            "aud_int": np.array(A, np.float32), "vis_int": np.array(V, np.float32)}

# ONE clean test set. Every network, whatever noise it trained at, is tested and projected on this.
test_clean = make_split("test", 0.0)
print("clean test set:", test_clean["X"].shape)

## 3. Model (masked GRU, self-connections only)

In [ ]:
class MaskedGRU(nn.Module):
    def __init__(self, n_in=4, hidden=2, n_out=4):
        super().__init__(); self.H = hidden; s = 1.0/math.sqrt(hidden)
        self.weight_ih = nn.Parameter(torch.empty(3*hidden, n_in).uniform_(-s, s))
        self.weight_hh = nn.Parameter(torch.empty(3*hidden, hidden).uniform_(-s, s))
        self.bias_ih   = nn.Parameter(torch.empty(3*hidden).uniform_(-s, s))
        self.bias_hh   = nn.Parameter(torch.empty(3*hidden).uniform_(-s, s))
        self.readout   = nn.Linear(hidden, n_out)
        self.register_buffer("mask", torch.eye(hidden).repeat(3, 1))
    def masked_hh(self): return self.weight_hh * self.mask
    def _run(self, x):
        B, Tt, _ = x.shape; H = self.H; Whh = self.masked_hh()
        Wir, Wiz, Win = self.weight_ih[:H], self.weight_ih[H:2*H], self.weight_ih[2*H:]
        Whr, Whz, Whn = Whh[:H], Whh[H:2*H], Whh[2*H:]
        bir, biz, bin_ = self.bias_ih[:H], self.bias_ih[H:2*H], self.bias_ih[2*H:]
        bhr, bhz, bhn = self.bias_hh[:H], self.bias_hh[H:2*H], self.bias_hh[2*H:]
        h = x.new_zeros(B, H); hs = []
        for t in range(Tt):
            xt = x[:, t, :]
            r = torch.sigmoid(xt @ Wir.T + bir + h @ Whr.T + bhr)
            z = torch.sigmoid(xt @ Wiz.T + biz + h @ Whz.T + bhz)
            n = torch.tanh(xt @ Win.T + bin_ + r * (h @ Whn.T + bhn))
            h = (1 - z) * n + z * h; hs.append(h)
        return torch.stack(hs, 1)
    def forward(self, x): return self.readout(self._run(x.transpose(1, 2)))
    def hidden_states(self, X):
        self.eval()
        with torch.no_grad(): h = self._run(torch.from_numpy(X).transpose(1, 2))
        return h.numpy()

## 4. Configuration

Two noise levels only, the extremes from notebook 14. `SAVE_DIR` lets the trained networks be reused, so
the figures can be redrawn later without retraining.

In [ ]:
HIDDEN = 2
SEEDS  = [0, 1, 2, 3, 4]
NOISE_LOW, NOISE_HIGH = 0.0, 2.0
FOCUS_SEED = 4                    # most collinear at low noise, so the largest expected change
N_EPOCHS = 50; LR = 1e-3; BATCH = 64
FIT_FROM = T_ON
SAVE_DIR = OUT_DIR / "models"; SAVE_DIR.mkdir(exist_ok=True)
REUSE_SAVED = True                # set False to force retraining
print("training noise levels: %.2f and %.2f, seeds %s, focus seed %d" % (NOISE_LOW, NOISE_HIGH, SEEDS, FOCUS_SEED))

## 5. Train (or reload), then measure the task axes on the clean test set

In [ ]:
def train_model(train_set, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = MaskedGRU(4, HIDDEN, 4).to(device)
    loader = DataLoader(TensorDataset(torch.from_numpy(train_set["X"]), torch.from_numpy(train_set["y"])),
                        batch_size=BATCH, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=LR); loss_fn = nn.CrossEntropyLoss()
    for _ in range(N_EPOCHS):
        model.train()
        for Xb, yb in loader:
            lo = model(Xb); B, Tt, C = lo.shape
            loss = loss_fn(lo.reshape(B*Tt, C), yb.unsqueeze(1).expand(B, Tt).reshape(B*Tt))
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); return model

def get_model(noise, seed):
    path = SAVE_DIR / ("masked_h%d_noise%.2f_seed%d.pt" % (HIDDEN, noise, seed))
    model = MaskedGRU(4, HIDDEN, 4).to(device)
    if REUSE_SAVED and path.exists():
        model.load_state_dict(torch.load(path)); model.eval(); return model, True
    model = train_model(make_split("train", noise), seed)
    torch.save(model.state_dict(), path)
    return model, False

def evaluate_clean(model):
    model.eval()
    with torch.no_grad(): pred = model(torch.from_numpy(test_clean["X"]))[:, -1, :].argmax(-1).numpy()
    ty, y = test_clean["types"], test_clean["y"]
    accs = {s: float((pred[ty==s]==y[ty==s]).mean()) for s in SUBTASKS}
    return pred, accs, float(np.mean([accs[s] for s in NONCONF]))

def fit_task_axes_clean(model):
    """Fit where/whether axes on the CLEAN test set, so all networks are measured identically."""
    Hs = model.hidden_states(test_clean["X"]); tt = np.arange(T) >= FIT_FROM
    ty, y4 = test_clean["types"], test_clean["y"]
    def build(mask_trials, pos):
        idx = np.where(mask_trials)[0]
        Xf = Hs[idx][:, tt, :].reshape(-1, Hs.shape[2])
        yf = np.repeat((y4[idx] == pos).astype(int), tt.sum())
        return Xf, yf
    Xw, yw = build(np.isin(ty, LOC_FIT), 2)      # right vs left
    Xd, yd = build(np.isin(ty, DET_FIT), 1)      # det vs no det
    res = {}
    for name, (Xf, yf) in [("where", (Xw, yw)), ("whether", (Xd, yd))]:
        dec = LogisticRegression(max_iter=2000).fit(Xf, yf)
        w = dec.coef_[0].astype(np.float64); b = float(dec.intercept_[0])
        if (Xf[yf == 1] @ w).mean() < (Xf[yf == 0] @ w).mean(): w, b = -w, -b   # pin sign
        nrm = np.linalg.norm(w) + 1e-12
        res[name] = (w/nrm, b/nrm, float(dec.score(Xf, yf)))
    w1, b1, a1 = res["where"]; w2, b2, a2 = res["whether"]
    ang = float(np.degrees(np.arccos(np.clip(abs(w1 @ w2), 0, 1))))
    return dict(w_where=w1, w_whether=w2, b_where=b1, b_whether=b2,
                acc_where=a1, acc_whether=a2, angle=ang)

def alignment_to_units(w):
    a = np.degrees(np.arctan2(abs(w[1]), abs(w[0]))); return float(min(a, 90 - a))

def project_clean(model, axes):
    Hs = model.hidden_states(test_clean["X"])
    return np.stack([Hs @ axes["w_where"], Hs @ axes["w_whether"]], -1)

store = {}
t0 = time.time()
for noise in [NOISE_LOW, NOISE_HIGH]:
    for s in SEEDS:
        m, cached = get_model(noise, s)
        a = fit_task_axes_clean(m)
        _, accs, nc = evaluate_clean(m)
        store[(noise, s)] = dict(model=m, axes=a, nonconf=nc, det_multi=accs["det_multisensory"],
                                 align_where=alignment_to_units(a["w_where"]),
                                 align_whether=alignment_to_units(a["w_whether"]))
        print("train noise %.2f seed %d%s: angle %5.1f deg  align(w/d) %4.1f/%4.1f  clean nonconf %.3f  dec %.3f/%.3f"
              % (noise, s, " [cached]" if cached else "", a["angle"],
                 store[(noise, s)]["align_where"], store[(noise, s)]["align_whether"],
                 nc, a["acc_where"], a["acc_whether"]), flush=True)
print("done in %.1f min" % ((time.time()-t0)/60))

## 6. Plot helpers

In [ ]:
def subtask_style(s):
    if s == "det_multisensory":   return {"color": "#8c3b00", "ls": "--", "lw": 2.0}
    if s.startswith("det"):       return {"color": "#e0852e", "ls": "-",  "lw": 1.8}
    if "conflict" in s:           return {"color": "#12355b", "ls": "--", "lw": 2.0}
    return {"color": "#5b9bd5", "ls": "-", "lw": 1.8}

def draw_axes_frame(ax, axes, Z, quadrants=False):
    xw, yd = -axes["b_where"], -axes["b_whether"]
    px = 0.15*max(np.ptp(Z[:,:,0]), 1e-6); py = 0.15*max(np.ptp(Z[:,:,1]), 1e-6)
    ax.axvline(xw, color="0.65", lw=0.9, ls=":"); ax.axhline(yd, color="0.65", lw=0.9, ls=":")
    ax.set_xlim(Z[:,:,0].min()-px, Z[:,:,0].max()+px)
    ax.set_ylim(Z[:,:,1].min()-py, Z[:,:,1].max()+py)
    if quadrants:
        ax.set_xlabel("z$_{where}$   (left \u2190 \u2192 right)")
        ax.set_ylabel("z$_{whether}$   (no det \u2193 \u2191 det)")

def plot_taskspace(ax, model, axes, title, legend=False):
    Z = project_clean(model, axes)
    draw_axes_frame(ax, axes, Z)
    for sub in SUBTASKS:
        idx = np.where(test_clean["types"] == sub)[0]
        tr = Z[idx].mean(0); st = subtask_style(sub)
        ax.plot(tr[:,0], tr[:,1], color=st["color"], ls=st["ls"], lw=st["lw"], alpha=0.9,
                label=sub if legend else None)
        ax.scatter(*tr[-1], color=st["color"], s=42, marker="*", edgecolor="k", lw=0.4, zorder=6)
    ax.scatter(*Z[:,0,:].mean(0), color="k", s=18, zorder=7)
    ax.set_title(title, fontsize=10)
    return Z

## Figure 1 — five seeds, low vs high training noise, all tested clean

Top row is trained without noise, bottom row trained at the highest noise. Every panel is tested and
projected on the **same clean trials**, so the only difference between rows is what the network learned.
The angle is printed in each title. The expectation from notebook 14 is that the top row is mixed, some
seeds orthogonal and some collinear, while the bottom row is consistently orthogonal.

In [ ]:
fig, axs = plt.subplots(2, len(SEEDS), figsize=(4.1*len(SEEDS), 8.4))
for r, noise in enumerate([NOISE_LOW, NOISE_HIGH]):
    for c, s in enumerate(SEEDS):
        d = store[(noise, s)]
        plot_taskspace(axs[r, c], d["model"], d["axes"],
                       "seed %d   angle %.0f\u00b0" % (s, d["axes"]["angle"]),
                       legend=(r == 0 and c == 0))
    axs[r, 0].set_ylabel("trained at noise %.1f\n\nz$_{whether}$" % noise, fontsize=10)
for c in range(len(SEEDS)):
    axs[1, c].set_xlabel("z$_{where}$", fontsize=9)
h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=6, fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.04))
fig.suptitle("Task space, all tested on clean trials: low (top) vs high (bottom) training noise", fontsize=13)
plt.tight_layout(rect=[0, 0.02, 1, 0.95])
plt.savefig(OUT_DIR / "noise_geometry_5seeds.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 2 — seed 4 close up

The clearest single comparison. This seed was the most collinear at low noise, so if training noise pushes
the code towards two separate dimensions, this is where the change should be unmistakable.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13.5, 6.2))
for ax, noise in zip(axs, [NOISE_LOW, NOISE_HIGH]):
    d = store[(noise, FOCUS_SEED)]
    Z = plot_taskspace(ax, d["model"], d["axes"],
                       "trained at noise %.1f   \u2014   angle %.1f\u00b0, alignment %.1f\u00b0"
                       % (noise, d["axes"]["angle"], d["align_where"]), legend=(noise == NOISE_LOW))
    draw_axes_frame(ax, d["axes"], Z, quadrants=True)
    # draw the two coding axes themselves, as arrows from the origin
    sc = 0.45*max(np.ptp(Z[:,:,0]), np.ptp(Z[:,:,1]))
    ax.annotate("", xy=(sc, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="#c0392b", lw=2))
    ax.annotate("", xy=(0, sc), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="#e0852e", lw=2))
    ax.text(sc, 0, " where", color="#c0392b", fontsize=9, va="center")
    ax.text(0, sc, " whether", color="#e0852e", fontsize=9, ha="left")
axs[0].legend(loc="center left", bbox_to_anchor=(-0.62, 0.5), fontsize=8, frameon=False)
fig.suptitle("Seed %d: the same network family trained without noise and at noise %.1f, both tested clean"
             % (FOCUS_SEED, NOISE_HIGH), fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(OUT_DIR / ("noise_geometry_seed%d.png" % FOCUS_SEED), dpi=150, bbox_inches="tight")
plt.show()

## Figure 3 — every seed's change, and the convergence

A paired plot. Each line is one seed moving from low to high training noise. Two things to read: whether
the lines generally go **up**, which is the effect of noise, and whether they **converge**, which is the
more interesting claim. At low noise the seeds are spread across the whole range, so initialisation
decides the solution. If high noise collapses them onto a narrow band near 90 degrees, then noise removes
the multiplicity of solutions and forces one particular organisation.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5.2))

# left: angle
ax = axs[0]
lo = [store[(NOISE_LOW, s)]["axes"]["angle"] for s in SEEDS]
hi = [store[(NOISE_HIGH, s)]["axes"]["angle"] for s in SEEDS]
for i, s in enumerate(SEEDS):
    ax.plot([0, 1], [lo[i], hi[i]], marker="o", lw=1.8, alpha=0.85, label="seed %d" % s)
ax.plot([0, 1], [np.mean(lo), np.mean(hi)], color="k", lw=3, ls="--", alpha=0.7, label="mean")
ax.axhline(90, color="green", ls=":", alpha=0.7); ax.text(1.02, 90, "orthogonal", color="green", fontsize=8, va="center")
ax.set_xticks([0, 1]); ax.set_xticklabels(["noise %.1f" % NOISE_LOW, "noise %.1f" % NOISE_HIGH])
ax.set_ylabel("angle between where and whether axes (deg)"); ax.set_ylim(-5, 100)
ax.set_title("angle: spread %.1f\u00b0 \u2192 %.1f\u00b0 (sd)" % (np.std(lo), np.std(hi)))
ax.grid(alpha=0.3, axis="y"); ax.legend(fontsize=8, frameon=False, loc="lower left")

# right: alignment to hidden-unit axes
ax = axs[1]
lo_a = [store[(NOISE_LOW, s)]["align_where"] for s in SEEDS]
hi_a = [store[(NOISE_HIGH, s)]["align_where"] for s in SEEDS]
for i, s in enumerate(SEEDS):
    ax.plot([0, 1], [lo_a[i], hi_a[i]], marker="s", lw=1.8, alpha=0.85, label="seed %d" % s)
ax.plot([0, 1], [np.mean(lo_a), np.mean(hi_a)], color="k", lw=3, ls="--", alpha=0.7, label="mean")
ax.axhline(45, color="purple", ls=":", alpha=0.7); ax.text(1.02, 45, "fully shared", color="purple", fontsize=8, va="center")
ax.axhline(0, color="0.6", ls=":", alpha=0.7); ax.text(1.02, 1, "one unit per task", color="0.5", fontsize=8, va="center")
ax.set_xticks([0, 1]); ax.set_xticklabels(["noise %.1f" % NOISE_LOW, "noise %.1f" % NOISE_HIGH])
ax.set_ylabel("alignment of the where axis to a hidden unit (deg)"); ax.set_ylim(-2, 50)
ax.set_title("do both units share the work?"); ax.grid(alpha=0.3, axis="y")

fig.suptitle("Effect of training noise on the learned solution, per seed (all tested clean)", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(OUT_DIR / "noise_geometry_paired.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Summary table

In [ ]:
print(f"{'seed':>5} {'angle low':>10} {'angle high':>11} {'change':>8} {'align low':>10} {'align high':>11} "
      f"{'clean acc low':>14} {'clean acc high':>15}")
print("-"*95)
for s in SEEDS:
    a, b = store[(NOISE_LOW, s)], store[(NOISE_HIGH, s)]
    print(f"{s:>5} {a['axes']['angle']:>9.1f}\u00b0 {b['axes']['angle']:>10.1f}\u00b0 "
          f"{b['axes']['angle']-a['axes']['angle']:>+7.1f}\u00b0 {a['align_where']:>9.1f}\u00b0 "
          f"{b['align_where']:>10.1f}\u00b0 {a['nonconf']:>14.3f} {b['nonconf']:>15.3f}")
lo = np.array([store[(NOISE_LOW, s)]["axes"]["angle"] for s in SEEDS])
hi = np.array([store[(NOISE_HIGH, s)]["axes"]["angle"] for s in SEEDS])
print("\nmean angle: %.1f\u00b0 -> %.1f\u00b0   (change %+.1f\u00b0)" % (lo.mean(), hi.mean(), hi.mean()-lo.mean()))
print("spread (sd): %.1f\u00b0 -> %.1f\u00b0" % (lo.std(), hi.std()))
if hi.std() < 0.6*lo.std():
    print("-> the seeds converge: high training noise narrows the range of solutions found.")
if hi.mean() > lo.mean() + 5:
    print("-> higher training noise pushes the code towards orthogonal where/whether axes.")
print("\nclean non-conflict accuracy: %.3f -> %.3f"
      % (np.mean([store[(NOISE_LOW, s)]['nonconf'] for s in SEEDS]),
         np.mean([store[(NOISE_HIGH, s)]['nonconf'] for s in SEEDS])))

## Notes

- **Every network is tested and projected on the same clean trials**, whatever noise it trained at, so a
  difference between rows or between panels is a difference in what was learned rather than a difference in
  what was shown at test.
- Only the two extreme noise levels are used, because the summary measures in notebook 14 change very little
  between 1.0 and 1.5, so the extremes carry the effect without crowding the figure.
- Seed 4 is singled out because it was the most collinear at low noise, so it has the most room to change.
  Seed 0 started near orthogonal and can only gain a few degrees.
- The clean accuracy column should be read alongside the geometry. If a high-noise network has poor clean
  accuracy it has not really learned the task, and its axes describe very little.
- Trained networks are cached in `noise_geometry_v8/models`, so the figures can be redrawn without retraining.